# DFU-PolarMorphNet — complete resumable experiment

One Markdown cell and one executable code cell. The cell installs dependencies, clones the GitHub project, mounts Drive, audits the dataset, resumes group-safe five-fold training, shows fold-wise live tables/figures, calibrates each fold, saves `.pt`/CSV/PKL artifacts, performs small XAI/robustness analyses, and exports GitHub-safe results. For multiple Colab accounts, use the same shared Drive root and RUN_ID while assigning disjoint PREFERRED_FOLDS.


In [ ]:
# ================= USER SETTINGS =================
MODE = "train"  # train | artifacts | upload
RUN_ID = ""      # First account: leave empty. Other accounts: paste the same printed RUN_ID.
PREFERRED_FOLDS = ""  # Examples: account-A "1,2"; account-B "3,4"; account-C "5"
DRIVE_ROOT = "/content/drive/MyDrive/DFU-PolarMorphNet"
REPO_REF = "q1-posthoc-corrections-20260801"  # Change to main after the PR is merged.
FORCE_RETRAIN = False
# =================================================

import os, pathlib, runpy, shutil, subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=1.0.9", "kagglehub>=0.3", "ImageHash>=4.3",
    "scikit-learn>=1.5", "scipy>=1.13", "matplotlib>=3.9",
    "pandas>=2.2", "Pillow>=10.4"
], check=True)

repo_dir = pathlib.Path("/content/DFU-ImageGuard")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_REF,
    "https://github.com/AzizulHakim00/DFU-ImageGuard.git", str(repo_dir)
], check=True)

os.environ["DFU_MODE"] = MODE
os.environ["DFU_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["DFU_PREFERRED_FOLDS"] = PREFERRED_FOLDS
if RUN_ID.strip():
    os.environ["DFU_RUN_ID"] = RUN_ID.strip()
else:
    os.environ.pop("DFU_RUN_ID", None)

script = repo_dir / "polarmorphnet" / "DFU_PolarMorphNet_AllInOne.py"
argv = [str(script), "--mode", MODE, "--drive-root", DRIVE_ROOT]
if RUN_ID.strip():
    argv += ["--run-id", RUN_ID.strip()]
if PREFERRED_FOLDS.strip():
    argv += ["--preferred-folds", PREFERRED_FOLDS.strip()]
if FORCE_RETRAIN:
    argv += ["--force-retrain"]
sys.argv = argv
runpy.run_path(str(script), run_name="__main__")
